# Baseline de documentação de código legado

Executa o benchmark inicial em PHP, Python, JavaScript e SQL.

Antes de começar, selecione **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Ative uma GPU no ambiente de execução do Colab.'
print('GPU:', torch.cuda.get_device_name(0))

## Enviar o projeto

Envie `Celx-colab-v3.zip`. O nome é validado pelo conteúdo, então versões futuras também poderão ser usadas.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
import os

uploaded = files.upload()
zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_files, (
    'Envie o arquivo dist/Celx-colab.zip, e não scripts/package_colab.ps1.'
)
archive_name = zip_files[0]
project_root = Path('/content/legacy-doc-project')
if project_root.exists():
    shutil.rmtree(project_root)
project_root.mkdir(parents=True)
shutil.unpack_archive(f'/content/{archive_name}', project_root)
candidates = list(project_root.rglob('model_candidates.yaml'))
assert candidates, 'configs/model_candidates.yaml não foi encontrado no ZIP.'
assert candidates[0].parent.name == 'configs', 'Estrutura inesperada no pacote.'
repo_dir = candidates[0].parent.parent
os.environ['PYTHONPATH'] = str(repo_dir)
print('Projeto:', repo_dir)

In [ ]:
%cd {repo_dir}
%pip install -q "transformers>=4.51" "accelerate>=1.0" "bitsandbytes>=0.45" "PyYAML>=6.0"
%pip install -q -e .

from legacy_doc.prompts import SYSTEM_PROMPT
print('Pacote legacy_doc carregado corretamente.')

## Teste inicial: Qwen3 1.7B

Esta célula baixa o modelo e executa os quatro casos do benchmark. Pode levar alguns minutos.

O aviso sobre requisições não autenticadas é informativo: o Qwen pode ser baixado sem token. Para obter limites maiores, crie um secret chamado `HF_TOKEN` no painel de chaves do Colab e execute a célula opcional abaixo.

In [ ]:
# Opcional: autenticação no Hugging Face usando um secret do Colab.
try:
    from google.colab import userdata
    from huggingface_hub import login
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Hugging Face autenticado.')
except Exception:
    print('HF_TOKEN não configurado; o Qwen será baixado sem autenticação.')

In [ ]:
import os
import subprocess
import sys

# Selecione os modelos que deseja comparar.
executar_qwen = True  # @param {type:"boolean"}
executar_ministral = True  # @param {type:"boolean"}

selecionados = [
    nome
    for nome, ativo in {
        'qwen3-1.7b': executar_qwen,
        'ministral3-3b': executar_ministral,
    }.items()
    if ativo
]
assert selecionados, 'Selecione pelo menos um modelo.'

for modelo in selecionados:
    print(f'\n=== Executando {modelo} ===')
    subprocess.run(
        [sys.executable, 'scripts/run_baseline.py', '--model', modelo],
        check=True,
        env=os.environ.copy(),
    )

In [ ]:
import json
from IPython.display import Markdown, display

result_files = sorted((repo_dir / 'evaluation/results/baseline').glob('*.jsonl'))
assert result_files, 'Nenhum resultado foi encontrado.'
for result_path in result_files:
    records = [json.loads(line) for line in result_path.read_text(encoding='utf-8').splitlines()]
    display(Markdown(f"# Modelo: {result_path.stem}"))
    for record in records:
        display(Markdown(f"## {record['id']} — {record['language']}"))
        display(Markdown(record['response']))
        print(f"Tempo: {record['elapsed_seconds']}s | Tokens: {record['generated_tokens']}")

In [ ]:
!python evaluation/score_baseline.py
!python evaluation/compare_models.py
import pandas as pd
from google.colab import files
scores_path = repo_dir / 'evaluation/results/human_scores.csv'
comparison_path = repo_dir / 'evaluation/results/model_comparison.csv'
display(pd.read_csv(comparison_path))
files.download(str(scores_path))
files.download(str(comparison_path))
for result_path in sorted((repo_dir / 'evaluation/results/baseline').glob('*.jsonl')):
    files.download(str(result_path))

## Observações sobre os candidatos

Qwen e Ministral podem ser baixados sem token. O Ministral é maior e pode consumir mais memória e tempo que o Qwen.

In [ ]:
print('Modelos disponíveis:')
print('- qwen3-1.7b: aberto, padrão recomendado para o primeiro teste')
print('- ministral3-3b: aberto, porém mais pesado')